# Web Skin 데이터셋 검증 전용
기존 `web_skin_processed` ZIP의 **Original / Augmented / train / val / test**를 검사합니다.
**학습·모델 로드·데이터 삭제·라벨 변경·재분할은 하지 않습니다. GPU나 기존 모델 ZIP이 필요 없습니다.**

- 파일 SHA-256: 이름이 달라도 내용이 완전히 같은 파일 탐지
- RGB 픽셀 해시: 저장 파일 내용이 달라도 디코딩된 픽셀이 같은 이미지 탐지
- Train / Validation / Test 간 중복과 클래스 간 라벨 충돌
- Original과 Augmented 평가셋의 라벨·수량·파일 내용 비교
- Augmented Train에 원본 Train이 포함되어 있는지 검사
- 이미지 읽기 오류, 해상도 분포, 같은 분할 안 중복, 원천 메타데이터 목록 기록
- 중복 사진 예시와 검사 ZIP을 Drive에 저장

클래스 순서는 기존 결과 JSON 기준 **건선, 아토피, 여드름, 정상, 주사**입니다.
사람·병변·촬영 세션·변형된 증강 파생 관계는 별도 원천 기록 검토가 필요합니다.
검사 통과는 모든 누수가 없다는 뜻이 아닙니다.

Colab에서 위에서 아래로 실행하세요. 문제가 발견되어도 보고서를 끝까지 저장합니다.


## 1. Drive 연결과 검사 도구
CPU 런타임으로 실행해도 됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install pandas pillow matplotlib tqdm

import hashlib
import json
import platform
import shutil
import stat
import uuid
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import PIL
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


## 2. 입력·결과 위치
경로를 비워두면 Drive에서 `web_skin_processed`로 시작하는 ZIP을 찾습니다. 여러 개면 경로 목록을 출력하며 멈춥니다. 정확한 파일을 `DATA_ZIP_OVERRIDE`에 입력한 뒤 다시 실행하세요.

In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_OVERRIDE = ''
CLASS_NAMES = ['건선', '아토피', '여드름', '정상', '주사']
CLASS_CODES = {name: f'C{i}' for i, name in enumerate(CLASS_NAMES)}
SPLITS = ('train', 'val', 'test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif', '.tif', '.tiff'}
TRAIN_LOADER_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
RUN_NAME = 'dataset_audit_' + RUN_ID
LOCAL_ROOT = Path('/content') / ('web_skin_' + RUN_NAME)
EXTRACT_ROOT = LOCAL_ROOT / 'dataset'
REPORT_DIR = MY_DRIVE / 'mediflow_experiments' / 'web_skin' / RUN_NAME
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
EXTRACT_ROOT.mkdir()
REPORT_DIR.mkdir(parents=True, exist_ok=False)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def save_json(name, value):
    (REPORT_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def save_csv(name, rows, columns):
    pd.DataFrame(rows, columns=columns).to_csv(REPORT_DIR / name, index=False, encoding='utf-8-sig')

def safe_extract(source, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(source) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError(f'안전하지 않은 ZIP 항목: {item.filename}')
        archive.extractall(destination)

print('검사 결과 폴더:', REPORT_DIR)
print(CLASS_CODES)


## 3. ZIP 확인·복사·압축 해제
원본 파일은 그대로 둡니다. 압축 해제 공간이 부족하면 중단합니다. 약 13GB인 데이터 ZIP의 경우 복사와 검사가 오래 걸릴 수 있습니다.

In [ ]:
if DATA_ZIP_OVERRIDE:
    candidates = [Path(DATA_ZIP_OVERRIDE)]
else:
    candidates = [p for p in MY_DRIVE.rglob('web_skin_processed*')
                  if p.is_file() and zipfile.is_zipfile(p)]
candidates = sorted({p.resolve() for p in candidates if p.is_file() and zipfile.is_zipfile(p)})
if len(candidates) != 1:
    save_json('input_path_error.json', {'candidates': [str(p) for p in candidates]})
    raise ValueError(f'검사할 ZIP을 하나로 지정하세요. DATA_ZIP_OVERRIDE 설정 필요: {candidates}')
DATA_ZIP_PATH = candidates[0]
with zipfile.ZipFile(DATA_ZIP_PATH) as archive:
    uncompressed_bytes = sum(item.file_size for item in archive.infolist())
required_bytes = DATA_ZIP_PATH.stat().st_size + uncompressed_bytes + 1024**3
free_bytes = shutil.disk_usage(LOCAL_ROOT).free
print('필요 공간(bytes):', required_bytes, '여유 공간(bytes):', free_bytes)
if required_bytes > free_bytes:
    raise RuntimeError('Colab 디스크 여유 공간이 부족합니다. 더 큰 디스크의 런타임이 필요합니다.')
LOCAL_ZIP = LOCAL_ROOT / 'input.zip'
shutil.copyfile(DATA_ZIP_PATH, LOCAL_ZIP)
DATA_SHA256 = sha256(LOCAL_ZIP)
if DATA_SHA256 != sha256(DATA_ZIP_PATH):
    raise IOError('Drive 원본과 Colab 복사본 해시가 다릅니다.')
safe_extract(LOCAL_ZIP, EXTRACT_ROOT)
save_json('source.json', {
    'domain': 'web_skin', 'source_zip': str(DATA_ZIP_PATH), 'sha256': DATA_SHA256,
    'zip_bytes': LOCAL_ZIP.stat().st_size, 'uncompressed_bytes': uncompressed_bytes,
    'run_id_utc': RUN_ID,
    'code_commit_at_creation': 'b5faa6d937229a49e9d62541a30e39f3b75a3c77',
    'code_state': '새 미커밋 노트북; code_snapshot.py에 실행 셀 기록',
    'environment': {'python': platform.python_version(), 'pandas': pd.__version__,
                    'pillow': PIL.__version__, 'matplotlib': matplotlib.__version__},
})
print('압축 해제 완료:', EXTRACT_ROOT)


## 4. 이미지 조사
파일명 대신 파일 내용과 실제 RGB 픽셀을 검사합니다. 추가 클래스나 빈 폴더도 보고합니다.

In [ ]:
roots, inventory, issues, metadata_files = {}, [], [], []
for path in EXTRACT_ROOT.rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.json', '.csv'}:
        metadata_files.append(str(path.relative_to(EXTRACT_ROOT)))
for kind in ('original', 'augmented'):
    matches = [p for p in EXTRACT_ROOT.rglob('*') if p.is_dir() and p.name.lower() == kind
               and all((p / split).is_dir() for split in SPLITS)]
    if len(matches) != 1:
        issues.append({'type': 'root_structure', 'detail': f'{kind}: {[str(p) for p in matches]}'})
        continue
    root = roots[kind] = matches[0]
    for split in SPLITS:
        classes = sorted(p.name for p in (root / split).iterdir() if p.is_dir())
        if classes != sorted(CLASS_NAMES):
            issues.append({'type': 'class_structure', 'detail': f'{kind}/{split}: {classes}'})
        stray = [p for p in (root / split).iterdir()
                 if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if stray:
            issues.append({'type': 'image_outside_class', 'detail': f'{kind}/{split}: {len(stray)}'})
        for cls in sorted(set(classes) | set(CLASS_NAMES)):
            paths = sorted(p for p in (root / split / cls).rglob('*')
                           if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
            if not paths:
                issues.append({'type': 'empty_class', 'detail': f'{kind}/{split}/{cls}'})
            for path in tqdm(paths, desc=f'{kind}/{split}/{CLASS_CODES.get(cls, cls)}'):
                relative_path = str(path.relative_to(EXTRACT_ROOT))
                row = {'kind': kind, 'split': split, 'class_name': cls, 'path': relative_path}
                try:
                    row['sha256'] = sha256(path)
                    with Image.open(path) as image:
                        row.update(width=image.width, height=image.height,
                                   mode=image.mode, frames=getattr(image, 'n_frames', 1))
                        rgb = image.convert('RGB')
                        row['pixel_sha256'] = hashlib.sha256(
                            str(rgb.size).encode('ascii') + b':' + rgb.tobytes()).hexdigest()
                    row['readable'] = True
                    if row['frames'] != 1:
                        issues.append({'type': 'multi_frame_image', 'detail': relative_path})
                except Exception as exc:
                    row['readable'] = False
                    issues.append({'type': 'unreadable_image', 'detail': f'{relative_path}: {exc}'})
                if path.suffix.lower() not in TRAIN_LOADER_EXTENSIONS:
                    issues.append({'type': 'training_loader_unsupported', 'detail': relative_path})
                inventory.append(row)
df = pd.DataFrame(inventory, columns=[
    'kind', 'split', 'class_name', 'path', 'sha256', 'pixel_sha256',
    'width', 'height', 'mode', 'frames', 'readable'])
df.to_csv(REPORT_DIR / 'image_inventory.csv', index=False, encoding='utf-8-sig')
save_json('metadata_inventory.json', {
    'files': sorted(metadata_files),
    'status': '파일 목록만 수집. 사람·병변·촬영 세션·증강 출처 대응 관계는 미검증',
})
print('조사한 이미지 파일 수:', len(df))


## 5. 중복·라벨·평가셋 일치 검사
보고서의 숫자는 구분해서 봐야 합니다. 중복 그룹 수, 파일 수, 분할쌍별 겹침 수는 서로 다르며 합쳐서 제거 장수라고 해석하면 안 됩니다. Original을 Augmented에 복사한 정상적인 관계는 분할 누수로 세지 않습니다.

In [ ]:
cross_split_rows, conflict_rows, within_rows, pair_rows, compare_rows = [], [], [], [], []
for key in ('sha256', 'pixel_sha256'):
    valid = df.dropna(subset=[key])
    for digest, group in valid.groupby(key):
        if group['split'].nunique() > 1:
            cross_split_rows.append({
                'hash_type': key, 'hash': digest, 'file_count': len(group),
                'splits': '|'.join(sorted(group['split'].unique())),
                'paths': '|'.join(group['path']),
            })
        if group['class_name'].nunique() > 1:
            conflict_rows.append({
                'hash_type': key, 'hash': digest, 'file_count': len(group),
                'classes': '|'.join(sorted(group['class_name'].unique())),
                'paths': '|'.join(group['path']),
            })
    for (kind, split, digest), group in valid.groupby(['kind', 'split', key]):
        if len(group) > 1:
            within_rows.append({'kind': kind, 'split': split, 'hash_type': key, 'hash': digest,
                                'file_count': len(group), 'paths': '|'.join(group['path'])})
    for kind in ('original', 'augmented'):
        for left, right in (('train', 'val'), ('train', 'test'), ('val', 'test')):
            a = set(valid[(valid.kind == kind) & (valid.split == left)][key])
            b = set(valid[(valid.kind == kind) & (valid.split == right)][key])
            pair_rows.append({'kind': kind, 'hash_type': key, 'pair': left + '_' + right,
                              'shared_hash_groups': len(a & b)})

def counted(kind, split, key='sha256'):
    part = df[(df.kind == kind) & (df.split == split)].dropna(subset=[key])
    return Counter(zip(part['class_name'], part[key]))

if len(roots) == 2:
    for split in ('val', 'test'):
        a, b = counted('original', split), counted('augmented', split)
        compare_rows.append({'check': 'evaluation_multiset_equal', 'split': split,
                             'passed': a == b, 'missing': sum((a-b).values()),
                             'extra': sum((b-a).values())})
        if a != b:
            issues.append({'type': 'evaluation_mismatch', 'detail': split})
    missing = counted('original', 'train') - counted('augmented', 'train')
    compare_rows.append({'check': 'original_train_contained', 'split': 'train',
                         'passed': not missing, 'missing': sum(missing.values()), 'extra': None})
    if missing:
        issues.append({'type': 'missing_train_original', 'detail': str(sum(missing.values()))})
if cross_split_rows:
    issues.append({'type': 'cross_split_duplicates', 'detail': 'cross_split_duplicates.csv 참조'})
if conflict_rows:
    issues.append({'type': 'label_conflicts', 'detail': 'label_conflicts.csv 참조'})

save_csv('cross_split_duplicates.csv', cross_split_rows,
         ['hash_type', 'hash', 'file_count', 'splits', 'paths'])
save_csv('label_conflicts.csv', conflict_rows,
         ['hash_type', 'hash', 'file_count', 'classes', 'paths'])
save_csv('within_split_duplicates.csv', within_rows,
         ['kind', 'split', 'hash_type', 'hash', 'file_count', 'paths'])
save_csv('split_overlap_counts.csv', pair_rows, ['kind', 'hash_type', 'pair', 'shared_hash_groups'])
save_csv('dataset_consistency.csv', compare_rows, ['check', 'split', 'passed', 'missing', 'extra'])
save_csv('audit_issues.csv', issues, ['type', 'detail'])
counts = df.groupby(['kind', 'split', 'class_name']).size().rename('count').reset_index()
counts.to_csv(REPORT_DIR / 'dataset_counts.csv', index=False, encoding='utf-8-sig')
resolution = df.groupby(['kind', 'split', 'width', 'height']).size().rename('count').reset_index()
resolution.to_csv(REPORT_DIR / 'resolution_counts.csv', index=False)
summary = {
    'domain': 'web_skin', 'data_sha256': DATA_SHA256, 'class_names': CLASS_NAMES,
    'status': 'issues_found' if issues else 'mechanical_checks_passed_with_limitations',
    'issue_count': len(issues), 'image_files': len(df), 'counts': counts.to_dict('records'),
    'cross_split_groups_by_hash_type': dict(Counter(r['hash_type'] for r in cross_split_rows)),
    'label_conflict_groups_by_hash_type': dict(Counter(r['hash_type'] for r in conflict_rows)),
    'within_split_duplicate_groups_by_hash_type': dict(Counter(r['hash_type'] for r in within_rows)),
    'metadata_file_count': len(metadata_files),
    'limitations': [
        '사람·병변·촬영 세션의 이미지 대응 정보 미검증',
        '증강 출처 기록의 연결 미검증; 변환된 파생본은 해시가 다를 수 있음',
        '재압축·밝기·회전·유사 장면 탐지는 미수행',
        '폴더 라벨의 의미상 정확성 및 복수 상태 동시 존재 여부 미검증',
        '이전 학습 당시 데이터 해시가 없어 원 학습 분할 동일성 입증 불가',
        '실제 웹캠 데이터 검증은 별도',
    ],
    'interpretation': 'SHA와 pixel 그룹은 같은 사례가 겹칠 수 있음. Original/Augmented 복사본도 포함하므로 합산 제거 수로 사용 금지.',
    'next_step': '보고서 검토 후 정제 또는 기존 모델 재현 여부 결정. 자동 학습하지 않음.',
}
save_json('audit_summary.json', summary)
display(counts)
display(pd.DataFrame(pair_rows))
display(pd.DataFrame(compare_rows))
print('검사 상태:', summary['status'], '문제 항목:', len(issues))
print('\n'.join(summary['limitations']))


## 6. 중복 예시와 수량 그래프
분할 간 중복과 라벨 충돌 예시를 최대 8쌍 보여줍니다. 그림은 C0~C4로 표시하고 실제 파일명은 `example_pairs.csv`에서 확인합니다. 사진이 없는 경우 숫자 그래프만 생성합니다.

In [ ]:
pairs, seen = [], set()
for key in ('sha256', 'pixel_sha256'):
    for digest, group in df.dropna(subset=[key]).groupby(key):
        if group['split'].nunique() <= 1 and group['class_name'].nunique() <= 1:
            continue
        readable = group[group['readable'] == True]
        if readable.empty:
            continue
        left = readable.iloc[0]
        alternatives = readable[(readable['split'] != left['split']) |
                                 (readable['class_name'] != left['class_name'])]
        if alternatives.empty:
            continue
        right = alternatives.iloc[0]
        identity = tuple(sorted([left['path'], right['path']]))
        if identity in seen:
            continue
        seen.add(identity)
        pairs.append((key, left, right))
        if len(pairs) >= 8:
            break
    if len(pairs) >= 8:
        break
example_rows = []
if pairs:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(10, 3.6 * len(pairs)), squeeze=False)
    for index, (key, left, right) in enumerate(pairs):
        example_rows.append({'case': index + 1, 'hash_type': key,
                             'left_path': left['path'], 'right_path': right['path']})
        for side, row in enumerate((left, right)):
            with Image.open(EXTRACT_ROOT / row['path']) as source:
                axes[index, side].imshow(source.convert('RGB'))
            label = CLASS_CODES.get(row['class_name'], 'unexpected class')
            axes[index, side].set_title(
                f"Case {index+1}: {row['kind']}/{row['split']}/{label}\n{key}", fontsize=9)
            axes[index, side].axis('off')
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'duplicate_examples.png', dpi=140)
    plt.show()
    plt.close(fig)
else:
    print('표시할 분할 간 중복/라벨 충돌 사진 쌍이 없습니다.')
save_csv('example_pairs.csv', example_rows, ['case', 'hash_type', 'left_path', 'right_path'])
if not counts.empty:
    plot_counts = counts.copy()
    plot_counts['class_name'] = plot_counts['class_name'].map(lambda x: CLASS_CODES.get(x, 'unexpected'))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, kind in zip(axes, ('original', 'augmented')):
        part = plot_counts[plot_counts.kind == kind]
        if not part.empty:
            part.pivot_table(index='class_name', columns='split', values='count',
                             aggfunc='sum', fill_value=0).plot.bar(ax=ax)
        ax.set(title='Web Skin ' + kind, ylabel='Image count', xlabel='Class code')
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'dataset_counts.png', dpi=180)
    plt.show()
    plt.close(fig)
save_json('class_code_mapping.json', CLASS_CODES)
print(CLASS_CODES)


## 7. 검사 결과 ZIP 저장
표시되는 ZIP을 내려받아 공유하세요. 원본 사진 전체는 포함하지 않고 검사 보고서와 최대 8쌍의 예시 그림만 묶습니다.

In [ ]:
(REPORT_DIR / 'code_snapshot.py').write_text(
    '\n\n# ---- cell ----\n\n'.join(get_ipython().history_manager.input_hist_raw), encoding='utf-8')
save_json('artifact_manifest.json', {
    p.name: {'sha256': sha256(p), 'bytes': p.stat().st_size}
    for p in REPORT_DIR.iterdir() if p.is_file() and p.name != 'artifact_manifest.json'
})
local_zip = Path(shutil.make_archive(str(LOCAL_ROOT / RUN_NAME), 'zip',
                                    root_dir=REPORT_DIR.parent, base_dir=REPORT_DIR.name))
destination = REPORT_DIR.parent / local_zip.name
if destination.exists():
    destination = destination.with_name(destination.stem + '_' + uuid.uuid4().hex[:8] + '.zip')
with local_zip.open('rb') as source, destination.open('xb') as output:
    shutil.copyfileobj(source, output)
if sha256(local_zip) != sha256(destination):
    raise IOError('검사 결과 ZIP 복사 해시가 다릅니다.')
print('검사 완료:', summary['status'])
print('공유할 결과 ZIP:', destination)
print('SHA-256:', sha256(destination))
print('학습 및 데이터 수정은 실행하지 않았습니다.')
